#### How to run this demo
> **simple and slow option** you can just click **Runtime → Run all** to run everything at once, or step through section by section.

> **optional fast option** - Before you run this section: switch to a faster processor (GPU)
This part runs much faster with a GPU. To turn it on:
> 1. At the very top of the page, click the **Runtime** menu. Click **Change runtime type**.
> 2. A small window will pop up. Under Hardware accelerator,
> 3. click T4 GPU.
> 4. Click Save.
> 5. Then, click **Runtime → Run all** as we've been doing it in other sessions.

The notebook may take a few seconds to reconnect — that's normal. Once it says "Connected" in the top-right corner, you're ready to run the cells below.

**After you've ran the notebook, remember to disconnect and delete the runtime**

> 1. Click the Runtime menu at the top.
> 2. Click Disconnect and delete runtime.
> 3. Confirm if it asks. That's it — the session is fully closed.

# Module 6 · Generative Models and the Boundaries of Authorship

This module focuses on one specific capability of modern AI: generation. AI systems can now produce text, images, and audio from plain-language instructions — accessible to anyone, requiring no technical expertise. That accessibility changes who can produce what, and at what volume. The legal questions this raises extend across multiple domains: authorship and originality, labor and displacement, authenticity and evidence, and the sustainability of the knowledge ecosystems that law has long sought to protect.

**Section 1 · From Noise to Form** — How generative models construct outputs from randomness, and what it means to "create" from statistical patterns.

**Section 2 · Prompting** — How instructions shape what a model produces, and whether directing a model's output constitutes a form of authorship.

**Section 3 · The Feedback Problem** — What happens when AI-produced output re-enters training data, and how the knowledge ecosystem degrades when human signal thins out.

**Section 4 · Style Mimicry** — How statistical patterns capture and reproduce an author's voice, and whether the style/expression distinction still holds at machine scale.

In [ ]:
#@title First, let us setup system, which would take a while {display-mode: "form"}
!pip install torch torchvision pillow ipywidgets imageio pillow-heif --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

import re, random, requests
from collections import defaultdict

import tensorflow as tf
from PIL import Image
import os, io, sys, shutil
import urllib.request
import textwrap

from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Input

from transformers import AutoTokenizer, AutoModelForCausalLM, T5ForConditionalGeneration

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef

import scipy.ndimage

import ipywidgets as widgets
from IPython.display import display, clear_output

import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
import transformers
transformers.logging.set_verbosity_error()


base_url = 'https://raw.githubusercontent.com/WeihaoGe1009/ai-history-for-ip-scholars/main/06_generative_models'


## Section 1. From Compression to Creation: Sampling from Noise

🔍 **The idea**

Now we would like to revisit module 4 about autoencoder and its applications. Those models we've seen *reconstruct*: you give them an image, they compress it and rebuild it. This section is about models that start from pure random noise and *shape* it, step by step, into something that looks real and new. This is the core move behind modern image generators, and it's what separates generation from reconstruction.

💡 **Why it matters**

These outputs can look strikingly original, but they aren't made from nothing. The model shapes that noise using patterns it learned from its training corpus, such as images, videos, literature, etc. These may include copyrighted works or an artist's distinctive style. So the question of where a generated image really "comes from" is not just technical. It's a legal and ethical one.

✅ **What the demo shows**

We'll present a generation process that builds a digit up from noise. You'll see the intermediate steps, so you can watch an image gradually emerge rather than just appear.

However, training a real diffusion model needs far more computing power than a demo like this allows. So what you'll see here is a lightweight illusgtration. It captures the *feel* of diffusion (starting from noise, refining step by step) without the heavy training behind it. The goal isn't a perfect diffusion model. It's to make one point clear: **generation is not reconstruction.**

## Demo 1. Turning Random Noise into Digits
In this demo, we will use a Variational Autoencoder (VAE) model, which is pretraine with the MNIST Dataset (the digitized hand-written number images we've encountered in Module 2). This autoencoder already "learns the features" of the hand-written digits, such as curves, turns, circles, and maybe straight lines. We will give it a pure random noise, and let the autoencoder create some "hand"-written numbers.

In [ ]:
#@title Step 1. Load a pre-trained VAE Model trained with MNIST Dataset {display-mode: "form"}
latent_dim = 10  # for easy visualization
# Encoder
encoder_inputs = layers.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation='relu', strides=2, padding='same')(encoder_inputs)
x = layers.Conv2D(64, 3, activation='relu', strides=2, padding='same')(x)
x = layers.Flatten()(x)
x = layers.Dense(16, activation='relu')(x)
z_mean = layers.Dense(latent_dim, name='z_mean')(x)
z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=(tf.shape(z_mean)[0], latent_dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon
z = layers.Lambda(sampling)([z_mean, z_log_var])
encoder = models.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
# Decoder
latent_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(7 * 7 * 64, activation='relu')(latent_inputs)
x = layers.Reshape((7, 7, 64))(x)
x = layers.Conv2DTranspose(64, 3, activation='relu', strides=2, padding='same')(x)
x = layers.Conv2DTranspose(32, 3, activation='relu', strides=2, padding='same')(x)
decoder_outputs = layers.Conv2DTranspose(1, 3, activation='sigmoid', padding='same')(x)
decoder = models.Model(latent_inputs, decoder_outputs, name="decoder")
# VAE Model
class VAE(tf.keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
    def train_step(self, data):
        if isinstance(data, tuple): data = data[0]
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(
                tf.keras.losses.binary_crossentropy(data, reconstruction)
            ) * 28 * 28
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )
            total_loss = reconstruction_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        return {"loss": total_loss, "reconstruction_loss": reconstruction_loss, "kl_loss": kl_loss}
    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        return self.decoder(z)
weights_url = f'{base_url}/models/vae.weights.h5'
model_path = '/tmp/vae.weights.h5'
os.system(f"wget -q '{weights_url}' -O {model_path}")
vae = VAE(encoder, decoder)
_ = vae(tf.zeros((1, 28, 28, 1)))
vae.load_weights(model_path)

In [ ]:
#@title Step 2. With the Pre-trained VAE Model, generate a scribble number from pure noise (might take a while) {display-mode: "both"}

latent_dim = 10
num_steps = 20
plot_step_delta=5
steps_to_plot = range(0,num_steps+plot_step_delta,plot_step_delta)  # Adjust for visualization

# Step 1: Start from a raw noise image (same shape as input images)
img = np.random.normal(loc=0.3, scale=1.0, size=(1, 28, 28, 1)).clip(0, 1).astype("float32")

# Step 2: Evolve through VAE encode → decode → encode...
images = [img[0].squeeze()]  # first image: pure noise

for i in range(num_steps):
    _, _, z = encoder.predict(img, verbose=0)
    img = decoder.predict(z, verbose=0)
    images.append(img[0].squeeze())

# Step 3: Plot selected steps
fig, axes = plt.subplots(1, len(steps_to_plot), figsize=(12, 2))
for ax, step in zip(axes, steps_to_plot):
    ax.imshow(images[step], cmap='gray')
    ax.set_title(f"Step {step}")
    ax.axis('off')
plt.tight_layout()
plt.show()


In this section, we load a pre-trained model that has learned how handwritten digits look (from the MNIST dataset we met in Module 2). We then feed it completely random noise, and let the model try to imagine a digit from that chaos.

The first image shows pure noise. Then, step by step, the model tries to “guess” a digit based on what it has learned. While we expected a slow, gradual change, in reality, the output very quickly starts to look like a number.

If you try running this several times, the final result might mostly look like a 3, or something between a 4 and a 9. Occasionally, the shape might even get blurrier instead of clearer, showing that the model doesn’t always find a perfect pattern.

This demo shows how a trained model can turn nonsense into something meaningful — a small glimpse into how generative AI works behind the scenes.



## Demo 2: Diffusion model

The last demo shows how a model trained on handwritten digits can produce rough sketches starting from noise. While the outputs were simple, the process illustrated how generative models can create something from randomness.

Now, we shift to **diffusion models** — which use a different idea: instead of jumping directly from noise to image, they build the image step by step, gradually reducing uncertainty.

This idea of gradual refinement is central to how generative systems behave. Seeing how a model moves from many possibilities to a single outcome offers a way to reflect on questions of influence, control, and decision-making in the creative process.

Because of time constraints, we won't run an actual diffusion model here. Instead, the demo below uses simple illustrative code to show conceptually how a diffusion model works: you'll watch a rough cat shape gradually emerge out of pure noise. This is, in essence, what image generators like Midjourney and Stable Diffusion are doing. Other tools, like Nano Banana (Google n.d.) or ChatGPT Images (OpenAI n.d.), use a different technique called "autoregression," which builds the image piece by piece instead of refining it from noise, though it still relies on the same kind of trained autoencoder to handle the actual pixels.

In [ ]:
#@title Illustrative Animation of Diffusion Model Mechanism {display-mode: "form"}
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import time

# === Load and prepare the cat image ===
#cat_image = Image.open("AI-gen-cat.png").resize((256, 256)).convert("RGB")
cat_image_name = "AI-gen-cat.png"
cat_image_url = f'{base_url}/data/02_diffusion_model/{cat_image_name}'
cat_image_file = f'/tmp/{cat_image_name}'
os.system(f"wget -q '{cat_image_url}' -O {cat_image_file}")
cat_image = Image.open(cat_image_file).resize((256, 256)).convert("RGB")
image = np.asarray(cat_image).astype(np.float32) / 255.0
np.random.seed(42)
noise = np.random.rand(*image.shape)

# === Set up 1D probability distribution ===
x = np.linspace(-10, 10, 500)
centers = [-6, -2, 0, 2.5, 6]
initial_sigmas = [3, 3, 3, 3, 3]
final_sigmas = [2.5, 2.0, 0.5, 1.5, 2.2]

n_frames = 30
alphas = np.linspace(0.0, 1.0, n_frames)

def amplitude_schedule(frame_idx, total_frames, peak_frame, fade_after=12, floor=0.02):
    if frame_idx < peak_frame:
        return max(floor, frame_idx / peak_frame)
    else:
        fade = 1 - (frame_idx - peak_frame) / fade_after
        return max(floor, fade)

# === Output display setup ===
output = widgets.Output()
display(output)

# === Animation loop ===
for t in range(n_frames):
    alpha = alphas[t]
    blended_image = (1 - alpha) * noise + alpha * image

    # Compute evolving distribution
    y_total = np.zeros_like(x)
    for i, mu in enumerate(centers):
        sigma = initial_sigmas[i] + (final_sigmas[i] - initial_sigmas[i]) * (t / (n_frames - 1))
        amp = amplitude_schedule(t, n_frames, peak_frame=10 + i * 2, fade_after=10, floor=0.02)
        gaussian = amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
        y_total += gaussian
    y_total /= y_total.max() + 1e-9  # normalize

    # Draw side-by-side plots
    with output:
        clear_output(wait=True)
        fig, axs = plt.subplots(1, 2, figsize=(8, 4))

        # Left: 1D probability distribution
        axs[0].plot(x, y_total, color='black', lw=2)
        axs[0].set_title(f"Step {t+1}/{n_frames} — Distribution evolving")
        axs[0].set_ylim(0, 1.05)
        axs[0].set_xlabel("Latent space")
        axs[0].set_ylabel("Relative probability")
        axs[0].grid(True, linestyle="--", alpha=0.3)

        # Right: Denoising image
        axs[1].imshow(blended_image)
        axs[1].axis("off")
        axs[1].set_title("Denoising — Image emerging")

        plt.tight_layout()
        plt.show()

    time.sleep(0.15)

# === Final frame: hold ===
with output:
    clear_output(wait=True)
    fig, axs = plt.subplots(1, 2, figsize=(7, 4))

    axs[0].plot(x, y_total, color='black', lw=2)
    axs[0].set_title("Final Step — Model converges to one output")
    axs[0].set_ylim(0, 1.05)
    axs[0].set_xlabel("Latent space")
    axs[0].set_ylabel("Relative probability")
    axs[0].grid(True, linestyle="--", alpha=0.3)

    axs[1].imshow(image)
    axs[1].axis("off")
    axs[1].set_title("Final Denoised Image")

    plt.tight_layout()
    plt.show()


This is a purely illustrative animation that mimics the process by which a diffusion model gradually constructs an image from random noise. Though simplified, it captures the step-by-step nature of how structure emerges over time.

* On the left, a probability distribution begins broad and uncertain, with multiple peaks suggesting different possibilities. As the process unfolds, the distribution narrows, eventually settling into one clear outcome.

* On the right, an image slowly takes shape from pure noise — gaining detail and coherence with each step.

While this animation is not derived from an actual diffusion model here, it conveys the core idea: generative models reduce uncertainty in stages, moving from open-ended ambiguity to a single output.

> *In fact, this image **is** pre-generated by an autoregressive model with ChatGPT Images(OpenAI 2026)*.

When directed, this process can be steered to mimic a particular style — or even imitate a person’s handwriting or voice. It can also produce something deceptively convincing, such as forged signatures or altered documents. But it might just as easily create something genuinely novel — combinations no one has thought to try before.

These possibilities are not limited to art. In other fields, the ability to generate unfamiliar but effective outputs can reshape practice. One of the most striking early examples of AI-generated surprise came from AlphaGo — not a generative model, but a reinforcement-learning system — whose now-legendary opening moves broke with convention and revealed new strategies that human players later studied and adopted.

This concludes our first section on how generative systems form content. In the next section, we’ll look at how that process can be guided — specifically, how prompts influence what a model chooses to build.

## Section 2. Prompting: Steering Without Coding

In this quick demo, we explore how small changes in prompts — such as tone or role — can shape the outputs of a generative model.

While the model and dataset remain the same, the phrasing of the prompt guides the style of response. You’ll see how shifting roles (teacher, journalist, activist) changes how a simple topic — penguin behavior — is presented. It will be a bit slow. If it is too slow, try



In [ ]:
#@title Demo 2. prompting different roles {display-mode: "form"}
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
#model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    torch_dtype=torch.bfloat16
)


prompts = [
    "Explain to children how penguins live, hunt, and socialize.",
    "Summarize penguin behavior in a factual news-reporting tone.",
    "Write the opening of a speech that calls the audience to pay attention to penguin behavior.",
]

print("DISCLAIMER:")
print(textwrap.fill(
    "All of the content below is automatically generated and purely illustrative — "
    "just like the evolving cat image from random noise. These texts are not factual, "
    "may contain hallucinations or nonsensical claims, and should not be taken seriously. "
    "Please focus on how the tone or style changes depending on the prompt.",
    width=80
))
print("\n" + "="*80 + "\n")

for prompt in prompts:
    #messages = [{"role": "user", "content": prompt}]
    messages = [
        {"role": "system", "content": "Respond directly with the requested text only. Do not explain your approach or give instructions."},
         {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        top_p=0.9,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.3
    )
    prompt_len = inputs["input_ids"].shape[-1]
    generated_ids = output_ids[0][prompt_len:]  # only the newly generated tokens
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    wrapped_output = textwrap.fill(generated_text, width=80)
    print(f"Prompt:\n{prompt.strip()}\n")
    print(f"Generated Output:\n{wrapped_output}")
    print("=" * 80)

While the outputs in this demo are entirely generated — and **should not** be taken as factual — they highlight how the same base model can adopt dramatically different voices and intentions, simply by changing the prompt.

In this case, the prompt steers tone. In more advanced use, prompts can shape structure, purpose, or even simulate distinct authorial styles. Prompting, as a practice, is increasingly used not just for generation, but for refinement, iteration, and interaction — a blend of creativity and control.

> *Fun Fact “This notebook, and in fact this entire 7-module demo series, was co-developed with ChatGPT and Claude prompts. So in a way... you’ve been interacting with prompt-powered creativity all along!”*

**Prompting can involve meaningful creative choices**. Even without changing the model itself, the output can vary significantly depending on how it’s prompted — something already under discussion in legal and policy circles when assessing authorship and contribution.

Until recently, producing text, images, or music at scale required either a large workforce or a long time. Prompting collapses both. A single person with a well-constructed instruction can now direct a model to produce outputs across registers, formats, and apparent purposes — outputs that previously required distinct human expertise to generate.

This shift matters beyond questions of authorship. It changes who can produce what, how much of it, and how fast. The downstream consequence of that change is what Section 3 examines: when output can be generated at this volume, some of it will find its way back into the data used to train the next generation of models — and something breaks.


## Section 3 The Feedback Problem


Sections 1 and 2 showed that generative models can produce text, images, and other content that some people find convincing — and others immediately recognize as artificial. Whether or not any individual output "passes" as human-made, one thing is clear: these systems now produce content at a scale and speed that human creators cannot match.

This volume has consequences. AI-produced content is already finding its way into the datasets used to train the next generation of AI models. When that happens, something breaks — not because the content is obviously wrong or mislabeled, but because it quietly misrepresents the world. It over-represents the typical, the average, the safe. It discards the rare, the ambiguous, or the unexpected.

Intuitively, people will think about AI-augmented misinformation. It is the "Garbage In - Garbage Out (GIGO)". For example, as many natural scientists concerned, that AI-generated animal behavior and anatomy are inaccurate and misleading.

However, as [Shumailov et.al.(2024)](https://www.nature.com/articles/s41586-024-07566-y) has pointed out, even if input data are accurate and error-free, AI-output quality is still doomed to degrade over the generations, even if nothing appears obviously wrong. We will see this in our demo.

*ref: Shumailov, Ilia, et al. "AI models collapse when trained on recursively generated data." Nature 631.8022 (2024): 755-759.*

In this demo, we utilize again the simple case we've utilized in Modulel 1. The simple classification problem that identifies whether a sentence is "legal" or "poetic". We use some sentences to train the Bag of Words model. Then, we let the model to make a prediction, and used the predicted the result as an input to train the model again over several generations.

First of all, we will intentionally "pollute" the training data set with mistakes from 0% (no mistake) to 40% (heavily corrupted). Then, we will focus on the deterioration of the error-free (0%) case.

In [ ]:
#@title Demo 3 - part 1: Generational accumulated deterioration from 0% to 40% input data mistakes {display-mode: "form"}


# ── Load data ─────────────────────────────────────────────────────────────────

input_file_name = "labeled_sentences.csv"
input_file_url = f'{base_url}/data/03_feedback/{input_file_name}'
input_file_local = f'/tmp/{input_file_name}'
os.system(f"wget -q '{input_file_url}' -O {input_file_local}")
df = pd.read_csv(input_file_local)
X  = df["Sentence"].values
y  = df["Label"].values

X_train_raw, X_test, y_train_clean, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ── Config ────────────────────────────────────────────────────────────────────

NOISE_LEVELS  = [0.0, 0.10, 0.20, 0.30, 0.40]
N_GENERATIONS = 8
TAIL_PCT      = 0.20
THRESHOLD_ACC = 0.80

COLORS = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6']
LABELS = [f"Noise {int(n*100)}%" for n in NOISE_LEVELS]

# ── Helpers ───────────────────────────────────────────────────────────────────

def inject_noise(y, noise_pct, seed=0):
    rng     = np.random.RandomState(seed)
    y_noisy = y.copy()
    idx     = rng.choice(len(y), size=int(len(y) * noise_pct), replace=False)
    y_noisy[idx] = 1 - y_noisy[idx]
    return y_noisy

def train_model(X_tr, y_tr):
    vec   = CountVectorizer(stop_words=None, min_df=1)
    X_bow = vec.fit_transform(X_tr)
    clf   = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_bow, y_tr)
    return clf, vec

def evaluate(clf, vec, X_te, y_te):
    X_bow  = vec.transform(X_te)
    y_pred = clf.predict(X_bow)
    probs  = clf.predict_proba(X_bow)
    return {
        "accuracy"        : accuracy_score(y_te, y_pred),
        "precision"       : precision_score(y_te, y_pred, zero_division=0),
        "recall"          : recall_score(y_te, y_pred, zero_division=0),
        "f1"              : f1_score(y_te, y_pred, zero_division=0),
        "mcc"             : matthews_corrcoef(y_te, y_pred),
        "mean_confidence" : probs.max(axis=1).mean(),
    }

def pool_vocab_size(X_pool):
    v = CountVectorizer(min_df=1)
    v.fit(X_pool)
    return len(v.vocabulary_)

def simulate_tail_loss(X_pool, y_pool, clf, vec, pct):
    X_bow      = vec.transform(X_pool)
    confidence = clf.predict_proba(X_bow).max(axis=1)
    keep_idx   = []
    extra_X, extra_y = [], []
    for cls in [0, 1]:
        cls_idx  = np.where(y_pool == cls)[0]
        cls_conf = confidence[cls_idx]
        n_tail   = max(1, int(len(cls_idx) * pct))
        drop_set = set(np.argsort(cls_conf)[:n_tail])
        dup_idx  = cls_idx[np.argsort(cls_conf)[-n_tail:]]
        keep_idx.extend([g for i, g in enumerate(cls_idx) if i not in drop_set])
        extra_X.extend(X_pool[dup_idx])
        extra_y.extend(y_pool[dup_idx])
    return (np.concatenate([X_pool[keep_idx], np.array(extra_X)]),
            np.concatenate([y_pool[keep_idx], np.array(extra_y)]))

# ── Run simulation ────────────────────────────────────────────────────────────

results = {}

for noise in NOISE_LEVELS:
    y_noisy = inject_noise(y_train_clean, noise, seed=42)
    pool_X  = X_train_raw.copy()
    pool_y  = y_noisy.copy()
    gen_history = []

    for gen in range(N_GENERATIONS + 1):
        clf, vec = train_model(pool_X, pool_y)
        metrics  = evaluate(clf, vec, X_test, y_test)
        vsize    = pool_vocab_size(pool_X)
        gen_history.append({"generation": gen, "vocab_size": vsize, **metrics})
        if gen < N_GENERATIONS:
            pool_X, pool_y = simulate_tail_loss(pool_X, pool_y, clf, vec, TAIL_PCT)

    results[noise] = gen_history

# ── Figure 1: Accuracy curves ─────────────────────────────────────────────────

def get_curve(noise, metric):
    return [r[metric] for r in results[noise]]

gens = list(range(N_GENERATIONS + 1))

fig, ax = plt.subplots(figsize=(9, 5))

for noise, color, label in zip(NOISE_LEVELS, COLORS, LABELS):
    ax.plot(gens, get_curve(noise, "accuracy"),
            marker='o', linewidth=2.2, color=color, label=label)

ax.axhline(THRESHOLD_ACC, color='gray', linestyle='--',
           linewidth=1.8, label='80% threshold')
ax.axhline(0.5, color='dimgray', linestyle=':',
           linewidth=2.2, label='50% (random guessing)')

ax.set_xlabel("Generation", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Model Collapse: Accuracy over Generations\n"
             "Does Better Input Data Help?",
             fontsize=13, fontweight='bold')
ax.set_xticks(gens)
ax.set_ylim(0.40, 1.02)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Better annotations won't save the day

The figure above demonstrates five separate runs, each starting with different levels of label errors.

**1. Better data delays collapse — but does not prevent it.**
The clean dataset (0% noise, green) starts at 96% accuracy and holds up
longer than the noisy ones. But by Generation 7-8, it has collapsed just
as completely. The endpoint is the same regardless of where you start.

**2. The time-to-failure gap is the real story.**
Clean data crosses the 80% threshold around Generation 4.
Data with 30–40% errors was already below 80% at Generation 0.

**3. All curves converge at the bottom.**
In the final generations, all
hovering near 50%, just like random guess. The model has lost all
meaningful signal, regardless of how good or bad its starting data was.

---

Now, Let's take a closer look on the 0% noise run.
The left panel shows accuracy collapsing over generations. The right panel, however, show the intrinsic mechanism why the model collapse: the total number of unique words in the training pool
shrinking as the models are trained on their own produced data.

Below the figure, we also compare the words the model associates most strongly with each class at Generation 0 versus the final generation.

In [ ]:
#@title Demo 3 - part 2: Under the hood: A Closer Look at Clean Data {display-mode: "form"}
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# results and helpers carried over from Cell 1

gens = list(range(N_GENERATIONS + 1))

# ── Re-run 0% noise capturing vocabulary details ──────────────────────────────

y_clean = inject_noise(y_train_clean, 0.0, seed=42)
pool_X  = X_train_raw.copy()
pool_y  = y_clean.copy()

gen0_history_detailed = []

correct_legal  = []
correct_poetic = []

for gen in range(N_GENERATIONS + 1):
    clf, vec = train_model(pool_X, pool_y)
    X_test_bow = vec.transform(X_test)
    y_pred     = clf.predict(X_test_bow)

    metrics  = evaluate(clf, vec, X_test, y_test)
    vsize    = pool_vocab_size(pool_X)

    feature_names = np.array(vec.get_feature_names_out())
    coef          = clf.coef_[0]
    top_legal  = feature_names[np.argsort(coef)[-10:][::-1]].tolist()
    top_poetic = feature_names[np.argsort(coef)[:10]].tolist()



    correct_legal.append(int(((y_pred == 1) & (y_test == 1)).sum()))
    correct_poetic.append(int(((y_pred == 0) & (y_test == 0)).sum()))


    gen0_history_detailed.append({
        "generation" : gen,
        "vocab_size"  : vsize,
        "top_legal"   : top_legal,
        "top_poetic"  : top_poetic,
        **metrics,
    })

    if gen < N_GENERATIONS:
        pool_X, pool_y = simulate_tail_loss(pool_X, pool_y, clf, vec, TAIL_PCT)

# ── Side-by-side figure ───────────────────────────────────────────────────────

acc_curve   = [r["accuracy"]   for r in gen0_history_detailed]
vocab_curve = [r["vocab_size"] for r in gen0_history_detailed]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: accuracy
ax1.plot(gens, acc_curve, marker='o', linewidth=2.5, color='#2ecc71')
ax1.axhline(0.80, color='gray',    linestyle='--', linewidth=1.8, label='80% threshold')
ax1.axhline(0.50, color='dimgray', linestyle=':', linewidth=2.2, label='50% (random guessing)')
ax1.set_xlabel("Generation", fontsize=12)
ax1.set_ylabel("Accuracy", fontsize=12)
ax1.set_title("Accuracy over Generations\n(0% Noise — Clean Data)", fontsize=12, fontweight='bold')
ax1.set_xticks(gens)
ax1.set_ylim(0.40, 1.02)
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Right: vocabulary size
ax2.plot(gens, vocab_curve, marker='s', linewidth=2.5, color='#8e44ad')
ax2.set_xlabel("Generation", fontsize=12)
ax2.set_ylabel("Unique Words in Training Pool", fontsize=12)
ax2.set_title("Vocabulary Size over Generations\n(0% Noise — Clean Data)", fontsize=12, fontweight='bold')
ax2.set_xticks(gens)
ax2.annotate(f"{vocab_curve[0]:,} words",
             xy=(0, vocab_curve[0]),
             xytext=(0.3, vocab_curve[0] - 100),
             fontsize=10, color='#8e44ad')
ax2.annotate(f"{vocab_curve[-1]:,} words",
             xy=(N_GENERATIONS, vocab_curve[-1]),
             xytext=(N_GENERATIONS - 2.8, vocab_curve[-1] + 90),
             fontsize=10, color='#8e44ad')
ax2.grid(True, alpha=0.3)

plt.suptitle("Clean Data Collapse: Accuracy and Vocabulary Narrow Together",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Vocabulary shift: derived dynamically from simulation output ───────────────

gen0 = gen0_history_detailed[0]
genN = gen0_history_detailed[-1]

print(f"── Vocabulary Shift: Generation 0 vs Generation {N_GENERATIONS} ──────────\n")

print("  Words most associated with LEGAL text:")
print(f"    Gen 0 : {gen0['top_legal']}")
print(f"    Gen {N_GENERATIONS} : {genN['top_legal']}")

# find words that disappeared and words that are new
legal_lost = [w for w in gen0['top_legal'] if w not in genN['top_legal']]
legal_new  = [w for w in genN['top_legal'] if w not in gen0['top_legal']]

if legal_lost:
    print(f"\n    Words that dropped out of the legal class top-10: {legal_lost}")
if legal_new:
    print(f"    New words that entered the legal class top-10:    {legal_new}")

print()
print("  Words most associated with POETIC text:")
print(f"    Gen 0 : {gen0['top_poetic']}")
print(f"    Gen {N_GENERATIONS} : {genN['top_poetic']}")

poetic_lost = [w for w in gen0['top_poetic'] if w not in genN['top_poetic']]
poetic_new  = [w for w in genN['top_poetic'] if w not in gen0['top_poetic']]

if poetic_lost:
    print(f"\n    Words that dropped out of the poetic class top-10: {poetic_lost}")
if poetic_new:
    print(f"    New words that entered the poetic class top-10:    {poetic_new}")

# flag cross-contamination: legal words appearing in poetic top-10 or vice versa
cross = [w for w in genN['top_poetic'] if w in gen0['top_legal']]
if cross:
    print(f"\n  ⚠ Cross-contamination: these words from the original legal")
    print(f"    vocabulary have moved into the poetic class by Gen {N_GENERATIONS}:")
    print(f"    {cross}")


We have observed that the vocabulary size decreased dramatically. For the legal class, the new top-10 words become more general. On the other hand, the poetic class vocabulary are showing some legal terminologies. Let's see how does the model correctly identify whether a sentence is "legal" or "poetic".

In [ ]:
#@title Demo 3 - part 3: Predictions over generations {display-mode: "form"}


# results and helpers carried over from Cells 1 and 3

gens = list(range(N_GENERATIONS + 1))

# Count of each class in the fixed test set
n_legal  = int((y_test == 1).sum())
n_poetic = int((y_test == 0).sum())


# ── Figure: grouped bar chart ─────────────────────────────────────────────────

x      = np.array(gens)
width  = 0.35

fig, ax = plt.subplots(figsize=(11, 5))

bars1 = ax.bar(x - width/2, correct_legal,  width, label=f'Legal sentences found (out of {n_legal})',
               color='#3498db', alpha=0.85)
bars2 = ax.bar(x + width/2, correct_poetic, width, label=f'Poetic sentences found (out of {n_poetic})',
               color='#e67e22', alpha=0.85)

# reference lines showing total available per class
ax.axhline(n_legal,  color='#3498db', linestyle='--', linewidth=1.2, alpha=0.5)
ax.axhline(n_poetic, color='#e67e22', linestyle='--', linewidth=1.2, alpha=0.5)

ax.set_xlabel("Generation", fontsize=12)
ax.set_ylabel("Sentences Correctly Identified", fontsize=12)
ax.set_title("How Many Sentences Did the Model Get Right — By Class?\n"
             "Clean Data (0% Noise)",
             fontsize=13, fontweight='bold')
ax.set_xticks(gens)
ax.set_ylim(0, max(n_legal, n_poetic) + 5)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# ── Plain-language summary ────────────────────────────────────────────────────

print(f"── Sentences correctly identified per generation ─────────────────────\n")
print(f"  {'Gen':>4}  {'Legal found':>14}  {'Poetic found':>14}")
print(f"  {'':>4}  {'(out of '+str(n_legal)+')':>14}  {'(out of '+str(n_poetic)+')':>14}")
print(f"  {'-'*40}")
for gen in gens:
    print(f"  {gen:>4}  {correct_legal[gen]:>14}  {correct_poetic[gen]:>14}")

print()
print(f"  By Generation {N_GENERATIONS}, the model correctly finds only "
      f"{correct_legal[-1]} of {n_legal} legal sentences")
print(f"  and {correct_poetic[-1]} of {n_poetic} poetic sentences.")
print()
print(f"  One class has effectively become invisible to the model.")

from this simulation, we can see that model totally "drops" one of the categories, and thus, it's predictions become trivial and unhelpful.

In [ ]:
#@title Demo 3 - part 4: confidence and accuracy {display-mode: "form"}
gens       = [r["generation"]       for r in gen0_history_detailed]
acc_curve  = [r["accuracy"]         for r in gen0_history_detailed]
conf_curve = [r["mean_confidence"]  for r in gen0_history_detailed]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(gens, acc_curve,  marker='o', linewidth=2.5, color='#2ecc71', label='Accuracy')
ax.plot(gens, conf_curve, marker='^', linewidth=2.5, color='#e74c3c', label='Confidence')
ax.fill_between(gens, acc_curve, conf_curve,
                where=[c > a for c, a in zip(conf_curve, acc_curve)],
                alpha=0.15, color='#e74c3c', label='Overconfidence gap')
ax.axhline(0.50, color='dimgray', linestyle=':', linewidth=2.2, label='50% — random guessing')
ax.set_xlabel("Generation", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Confidence vs Accuracy — Certainty and Correctness Come Apart",
             fontsize=12, fontweight='bold')
ax.set_xticks(gens)
ax.set_ylim(0.40, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(fontsize=10, loc='lower left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"  {'Gen':>4}  {'Accuracy':>10}  {'Confidence':>12}  {'Gap':>8}")
print(f"  {'-'*38}")
for gen, acc, conf in zip(gens, acc_curve, conf_curve):
    print(f"  {gen:>4}  {acc:>10.1%}  {conf:>12.1%}  {conf-acc:>+8.1%}")

#### The Model Grows more Certain and Less Correct

When we make a decision under uncertainty, we not only reach a conclusion, but also have a sense of how convinced we are. That sense can be strong or weak, no matter if we turn out to be right.

A classifier and other models, likewise, produces an internal measure of how strongly it favors one label over the alternative(s). This is called the **confidence score**. A score closer to 1 means the model finds the input overwhelmingly characteristic of that output, even if it is wrong or misleading.

 A model trained on a narrow set of examples will still report high confidence, because it has lost the ability to recognize when it is out of its depth.

In our example above, as training narrows over generations, accuracy falls. However, we found that confidence does not follow. The two curves come apart.


### Institutional Knowledge Degradation: A Dynamic Model

#### Research Question

When labor-intensive knowledge work is automated away, how does this affect the quality of information available to train subsequent AI systems? And what are the policy implications for systems designed to protect knowledge creation?

#### The Mechanism

This model tests a hypothesis about feedback loops in knowledge production:

1. Organizations employ workers at different skill levels who acquire expertise through structured experience over time.
2. When less-skilled positions are automated, the pathway for workers to gain experience is disrupted.
3. This reduces the supply of experienced workers available to supervise, evaluate, and provide feedback.
4. The quality of labeled data and feedback signals—essential inputs for training AI—degrades.
5. AI systems trained on degraded data perform worse.

The mechanism is not that individual workers become less competent, but that the institutional capacity to generate high-quality information degrades.

#### Model Components

**Workforce stratification**: Organizations maintain two skill tiers. Workers transition between tiers through experience accumulation. Experienced workers exit through retirement and attrition at constant rates. This is grounded in occupational mobility data.

**Automation trajectory**: Less-skilled positions are progressively automated. This is exogenous to the model—driven by business decisions and technology availability—not endogenous to knowledge quality.

**Information quality**: Information quality depends on two factors: (1) availability of experienced workers to supervise and evaluate, and (2) diversity of problems being solved by less-skilled workers. When less-skilled positions disappear, both factors degrade. Information quality is modeled as a function sensitive to the loss of either component.

**AI system performance**: AI systems are retrained periodically on newly available information. Performance depends on the quality of training data. When training data quality declines, system performance declines. This is empirically documented by Shumailov et al. (Nature, 2024) for recursive training on degraded synthetic data.

#### Empirical Grounding

The automation rate is calibrated to labor market data: SignalFire (2025) and Rezi.ai (2026) document approximately 50% decline in entry-level hiring across major sectors from 2019–2024. The model assumes this trend continues at a linear rate over 8 years to full automation.

Worker transition rates (experienced worker attrition at 5% annually; less-skilled to experienced transition at 8% annually) are consistent with Bureau of Labor Statistics occupational mobility data for professional occupations.

The mechanism linking information quality to AI performance is derived from Shumailov et al. (2024), which shows that models trained on progressively degraded synthetic data exhibit compounding performance loss.

#### Policy Implications

**On information as institutional product**: The simulation shows that information quality is not simply a function of database size or data collection methods. It depends on institutional capacity to generate, evaluate, and curate information. This suggests that policies regulating AI training data should account for the organizational structures producing that data.

**On knowledge production incentives**: If automation of less-skilled work degrades the information available for subsequent learning and AI training, there may be unintended negative externalities from labor-replacing automation. Policy frameworks designed to encourage knowledge production might need to account for these system-level effects.

**On creator economy sustainability**: The underlying mechanism—that experience accumulation requires structured learning opportunities—is relevant to debates about how creative and knowledge-intensive industries sustain themselves. If entry-level opportunities disappear, the pipeline of skilled creators narrows.

#### Sources

Polanyi, M. (1958). *Personal Knowledge*. University of Chicago Press.
- Foundational argument that expertise involves tacit, non-codifiable components acquired through practice.

Nonaka, I., & Takeuchi, H. (1995). *The Knowledge Creating Company*. Oxford University Press.
- Model of how organizations create knowledge through interaction between experienced and less-experienced workers.

Davenport, T. H., & Prusak, L. (1998). *Working Knowledge*. Harvard Business School Press.
- Documents mechanisms of knowledge loss in organizations and role of structured learning.

SignalFire. (2025). *State of Tech Talent 2025*.
- Documents 50% decline in entry-level hiring 2019–2024.

Rezi.ai. (2026). *The Crisis of Entry-Level Labor in the Age of AI*.
- Analysis of entry-level labor market disruption and institutional knowledge implications.

Shumailov, I., et al. (2024). "AI Models Collapse When Trained on Recursively Generated Data." *Nature*, 631(8022), 755–759.
- Empirical demonstration of performance degradation in models trained on progressively degraded synthetic data.

In [ ]:
#@title Demo 3 - part 5: Ecosystem Collapse Simulation {display-mode: "form"}
"""
Talent Pipeline + Training Data Degradation

Simulates: Junior→Senior pipeline collapse when AI replaces entry-level roles.
Result: Seniors retire without replacement → data quality degrades → model collapses.
"""



# ============================================================================
# PARAMETERS
# ============================================================================

# Initial populations
S_baseline = 500   # Baseline senior count
J_baseline = 1000  # Baseline junior count

# Dynamics
delta = 0.05       # Senior retirement rate (5% per year)
rho = 0.08         # Junior→Senior promotion rate (8% per year; reduced to slow replacement)
lambda_param = 1.5 # Mentorship starvation sensitivity (reduced to make q decay faster)
eta = 0.06         # Learning rate from new data (reduced; forgetting should dominate)
alpha = 1.2        # Vocabulary collapse exponent

# AI replacement timeline
t_start_ai = 0     # Year when AI replacement begins (year 0 = 2025)
T_replacement = 8  # Time to 100% replacement (8 years, faster)

# Model accuracy dynamics
M_baseline = 0.96  # Baseline accuracy (from Part A)
mu = 0.08          # ADJUSTED: Higher forgetting rate (8% per year without good data)

# Simulation length
n_years = 25

# ============================================================================
# INITIALIZE STATE
# ============================================================================

years = np.arange(n_years)
S = np.zeros(n_years)
J = np.zeros(n_years)
A = np.zeros(n_years)
J_eff = np.zeros(n_years)
q = np.zeros(n_years)
M = np.zeros(n_years)
V = np.zeros(n_years)

# Initial conditions
S[0] = S_baseline
J[0] = J_baseline
M[0] = M_baseline
V[0] = 1200  # Initial vocabulary size (from Part A)

# ============================================================================
# SIMULATION
# ============================================================================

for t in range(n_years - 1):

    # AI replacement fraction (linear ramp)
    A[t] = min(1.0, (t - t_start_ai) / T_replacement) if t >= t_start_ai else 0.0

    # Effective junior population (only non-AI juniors learn)
    J_eff[t] = J[t] * (1 - A[t])

    # Data quality: depends on senior supply and junior demand
    # Exponential starvation when J_eff is low
    senior_ratio = S[t] / S_baseline
    junior_ratio = J_eff[t] / J_baseline

    # Data quality decays if either seniors are scarce OR juniors are gone
    q[t] = senior_ratio * np.exp(-lambda_param * (1 - junior_ratio))

    # Clamp to [0, 1]
    q[t] = np.clip(q[t], 0, 1)

    # Update senior population
    # Seniors retire at rate delta; juniors get promoted proportionally to junior_eff
    S[t+1] = S[t] * (1 - delta) + rho * J_eff[t]

    # Update junior population (constant hiring, but need seniors to train them)
    # Simplified: assume junior inflow = constant, but quality of training depends on S
    # For simplicity, juniors stay constant in count (but their training quality degrades)
    J[t+1] = J[t]

    # Model accuracy: improves from fresh data, but degraded data amplifies failure
    # M(t+1) = M(t) * (1 - mu) + eta * q(t)^2
    # Poor data (low q) compounds: the quadratic term amplifies degradation
    M[t+1] = M[t] * (1 - mu) + eta * (q[t] ** 2)

    # Clamp accuracy to [0, 1]
    M[t+1] = np.clip(M[t+1], 0, 1)

    # Vocabulary size: collapses faster than accuracy
    # V(t) = V_baseline * M(t)^alpha
    V_baseline_current = 1200
    V[t+1] = V_baseline_current * (M[t+1] ** alpha)

# Handle final timestep
A[n_years-1] = min(1.0, (n_years - 1 - t_start_ai) / T_replacement) if n_years - 1 >= t_start_ai else 0.0
J_eff[n_years-1] = J[n_years-1] * (1 - A[n_years-1])
senior_ratio = S[n_years-1] / S_baseline
junior_ratio = J_eff[n_years-1] / J_baseline
q[n_years-1] = senior_ratio * np.exp(-lambda_param * (1 - junior_ratio))
q[n_years-1] = np.clip(q[n_years-1], 0, 1)

# ============================================================================
# PLOTTING
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Ecosystem Collapse: When AI Replaces the Junior Tier",
             fontsize=16, fontweight='bold', y=0.995)

# Color scheme
color_honeymoon = '#FFD700'
color_plateau = '#FFA500'
color_collapse = '#DC143C'

# Vertical lines marking phases
honeymoon_end = t_start_ai + 3
plateau_end = t_start_ai + 7

# --- Subplot 1: Senior Population ---
ax = axes[0, 0]
ax.plot(years, S, linewidth=2.5, color='#1f77b4', marker='o', markersize=4, label='Senior Population')
ax.axhline(S_baseline, color='gray', linestyle='--', alpha=0.5, label='Baseline')
ax.axvspan(t_start_ai, honeymoon_end, alpha=0.15, color=color_honeymoon, label='Honeymoon')
ax.axvspan(honeymoon_end, plateau_end, alpha=0.15, color=color_plateau, label='Plateau')
ax.axvspan(plateau_end, n_years, alpha=0.15, color=color_collapse, label='Collapse')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Number of Seniors', fontsize=11)
ax.set_title('Mentor Supply Collapse', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)
ax.legend(loc='best', fontsize=9)

# --- Subplot 2: Data Quality ---
ax = axes[0, 1]
ax.plot(years, q, linewidth=2.5, color='#2ca02c', marker='s', markersize=4)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, linewidth=1.5, label='Degradation Threshold')
ax.axvspan(t_start_ai, honeymoon_end, alpha=0.15, color=color_honeymoon)
ax.axvspan(honeymoon_end, plateau_end, alpha=0.15, color=color_plateau)
ax.axvspan(plateau_end, n_years, alpha=0.15, color=color_collapse)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Data Quality (0–1)', fontsize=11)
ax.set_title('Training Signal Degradation', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.grid(alpha=0.3)
ax.legend(loc='best', fontsize=9)

# --- Subplot 3: Model Accuracy ---
ax = axes[1, 0]
ax.plot(years, M, linewidth=2.5, color='#d62728', marker='^', markersize=4, label='Model Accuracy')
ax.axhline(M_baseline, color='gray', linestyle='--', alpha=0.5, label='Baseline')
ax.axhline(0.7, color='orange', linestyle=':', alpha=0.7, linewidth=1.5, label='Crisis Threshold (70%)')
ax.axvspan(t_start_ai, honeymoon_end, alpha=0.15, color=color_honeymoon)
ax.axvspan(honeymoon_end, plateau_end, alpha=0.15, color=color_plateau)
ax.axvspan(plateau_end, n_years, alpha=0.15, color=color_collapse)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Accuracy on Test Set', fontsize=11)
ax.set_title('Model Performance Collapse', fontsize=12, fontweight='bold')
ax.set_ylim([0.5, 1.0])
ax.grid(alpha=0.3)
ax.legend(loc='best', fontsize=9)

# --- Subplot 4: Vocabulary Size ---
ax = axes[1, 1]
ax.plot(years, V, linewidth=2.5, color='#9467bd', marker='D', markersize=4)
ax.axhline(1200, color='gray', linestyle='--', alpha=0.5, label='Baseline')
ax.axvspan(t_start_ai, honeymoon_end, alpha=0.15, color=color_honeymoon)
ax.axvspan(honeymoon_end, plateau_end, alpha=0.15, color=color_plateau)
ax.axvspan(plateau_end, n_years, alpha=0.15, color=color_collapse)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Vocabulary Size', fontsize=11)
ax.set_title('Knowledge Vocabulary Narrowing', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)
ax.legend(loc='best', fontsize=9)

plt.tight_layout()
plt.show()

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================

print("\n" + "="*70)
print("ECOSYSTEM COLLAPSE SIMULATION RESULTS")
print("="*70)

# Find time to crisis (when accuracy falls below 70%)
crisis_year = None
for i, acc in enumerate(M):
    if acc < 0.7 and crisis_year is None:
        crisis_year = years[i]

print(f"\nPhase 1: Honeymoon (Year 0–3)")
print(f"  Senior population: {S[0]:.0f} → {S[3]:.0f}")
print(f"  Data quality: {q[0]:.3f} → {q[3]:.3f}")
print(f"  Model accuracy: {M[0]:.4f} → {M[3]:.4f}")

print(f"\nPhase 2: Plateau (Year 3–7)")
print(f"  Senior population: {S[3]:.0f} → {S[7]:.0f}")
print(f"  Data quality: {q[3]:.3f} → {q[7]:.3f}")
print(f"  Model accuracy: {M[3]:.4f} → {M[7]:.4f}")

print(f"\nPhase 3: Collapse (Year 7+)")
print(f"  Senior population: {S[7]:.0f} → {S[-1]:.0f}")
print(f"  Data quality: {q[7]:.3f} → {q[-1]:.3f}")
print(f"  Model accuracy: {M[7]:.4f} → {M[-1]:.4f}")

print(f"\nCritical Metrics:")
print(f"  Time to crisis (accuracy < 70%): Year {crisis_year}")
print(f"  Final senior count: {S[-1]:.0f} ({S[-1]/S_baseline*100:.1f}% of baseline)")
print(f"  Vocabulary narrowing: {V[0]:.0f} → {V[-1]:.0f} tokens ({V[-1]/V[0]*100:.1f}%)")

print("\n" + "="*70)

# ============================================================================
# SAVE STATE DATA
# ============================================================================

import json

state_data = {
    'years': years.tolist(),
    'seniors': S.tolist(),
    'juniors': J.tolist(),
    'ai_replacement_fraction': A.tolist(),
    'data_quality': q.tolist(),
    'model_accuracy': M.tolist(),
    'vocabulary_size': V.tolist()
}

with open('ecosystem_state_data.json', 'w') as f:
    json.dump(state_data, f, indent=2)

## What this simulation shows

This demo models a hypothetical feedback loop: what happens to AI systems over time if the entry-level ("junior") jobs that train future experts get automated away. It runs a 25-year simulation and tracks four things, shown in the four graphs:

- **Mentor supply (top-left):** As senior workers retire and fewer juniors get promoted to replace them, the pool of experienced people who can train and supervise others shrinks.
- **Training signal quality (top-right):** With fewer mentors and fewer juniors solving real problems, the quality of the data used to train AI systems degrades — dropping past a "danger" threshold.
- **Model performance (bottom-left):** As the training data quality falls, the AI's accuracy declines, eventually crossing a crisis line (70%).
- **Knowledge breadth (bottom-right):** The range of things the system "knows" (its vocabulary) narrows even faster than accuracy falls.

The printout below the graphs breaks the 25 years into three phases — an early **honeymoon** period where things look fine, a **plateau** where cracks appear, and a **collapse** where quality drops sharply — and reports the year the model's accuracy falls below 70%.

**The takeaway:** the simulation illustrates an argument, not a prediction. Its point is that the quality of information available to train AI may depend on *human institutional structures* (like career ladders and mentorship), not just on how much data you collect. If the pipeline that produces experienced people breaks, the data — and the AI trained on it — could degrade too.

## Section 4. Style Mimicry: A Problem Across Creative Fields

Style has never been protected by copyright. The law draws a firm line between *expression* — the specific sentences, brushstrokes, or notes an author fixed in a work — and *style*, the recognizable manner in which they work. Style belongs to the commons: anyone may learn from it, be influenced by it, or work in it.

This principle is under pressure across every creative domain. In **visual art**, digital artist Greg Rutkowski's fantasy style was invoked over 400,000 times as a prompt on Stable Diffusion — generating a volume of work indistinguishable in feel from his own, without his consent or compensation [(Andersen v. Stability AI, N.D. Cal. 2023, ongoing)](www.courtlistener.com/docket/66732129/andersen-v-stability-ai-ltd/.). In **music**, AI composition tools Suno and Udio were sued by major record labels in 2024 for training on copyrighted recordings to capture the statistical patterns of musical genres and artist voices — with defendants arguing, consistent with existing doctrine, that "no company controls a genre or style of music" [(*American Bar Association*, 2025)](www.americanbar.org/groups/entertainment_sports/resources/entertainment-sports-lawyer/2025-fall/never-going-style-ai-generated-music-reliance-style-based-prompts-copyright-musical-genre/.). In **fashion**, AI-assisted design tools now generate silhouettes and prints "in the style of" living designers, raising questions courts have only begun to reach [(Oxford *Journal of Intellectual Property Law & Practice*, 2025)](https://academic.oup.com/jiplp/article/21/1/80/8329314). The U.S. Copyright Office's 2024–2025 AI reports address digital replicas of voice and likeness, but the broader question of extractable style remains legislatively open.

The common thread across all three fields is not copying — it is **extraction at scale**. A machine trained on an artist's body of work captures the statistical regularities that make their output recognizable, then reproduces those regularities on demand, in new works, at zero marginal cost. No specific protected expression is reproduced. The style — and the market it sustains — is what is displaced.

The premise underlying the style/expression distinction has always been implicit: that extracting and deploying a style requires human skill, time, and creative transformation. A student who spends years absorbing Dickens before writing their own novel is not a competitive threat to Dickens. A machine that reads his entire corpus in seconds and generates prose in his voice — at industrial scale — is a different kind of actor entirely. Whether the old doctrine adequately addresses this new actor is the question legislatures and courts are now beginning to confront.

---

Here, we present you with a demo about how image style transfer works. You can still observe some awkwardity that the images used as a reference were mainteined too much in the demo. However, modern generative models do the same at far greater fidelity, not limited to painting, but writing, composition, filming. Each style can be degradated to a couple of features represented by lists of numbers, where the computer programms can calculate at what spatial or temporal position, what is the probability of each value or value range, thus reproducing the "styles". For example, writers have their favorite own word or phrase combinations and sentence lengths; painters have their prefered palletes and strokes; animators have their habitulized paces and developments, etc. These, unfortunately, can all be learnt by AI models with ease.

**Try it:** Below, we use a model, a neural style transfer (Gatys *et al.*, 2015) with **VGG-19** features which is trained to extract image features. The structures are similar but with more layers to what we've learnt in Module 2. We would use it to make an arbitrary image mimick a style of Byzantine mosaic, the Great Wave by Hokusai, the Starry Night by Van Gogh, and the Composition 8 by Kandinsky. You could choose to upload your own image or let the system pick a random one from the STL-10 data set.

In [ ]:
#@title Let's get our pre-written codes and images ready for Demo 4. It will take a while. {display-mode: "form"}


### verifying GPU or CPU
_label = 'GPU' if torch.cuda.is_available() else 'CPU  (style transfer will take ~3-5 min)'
print(f'Running on: {_label}')

### getting the files and codes ready for section 06
LOCAL         = '/tmp/style_transfer_demo'
STYLE_LIBRARY = os.path.join(LOCAL, 'data/04_style_demo/style_library')
STL10_SUB_DIR = os.path.join(LOCAL, 'data/04_style_demo/stl10_sub')

if os.path.exists('/content/script'):
    shutil.rmtree('/content/script')
os.makedirs('/content/script')
open('/content/script/__init__.py', 'w').close()
print('Downloading demo scripts ...')
for _f in ['config.py', 'image_utils.py', 'vgg_extractor.py',
            'style_transfer.py', 'stl10_loader.py', 'ui_widgets.py', 'handlers.py']:
    os.system(f'wget -q {base_url}/script/{_f} -O /content/script/{_f}')

os.makedirs(STYLE_LIBRARY, exist_ok=True)
print('Downloading style images ...')
for _img in ['starry_night.jpg', 'great_wave.jpg',
              'kandinsky_comp8.jpg', 'byzantine_mosaic.jpeg']:
    os.system(f'wget -q {base_url}/data/04_style_demo/style_library/{_img}'
              f' -O {os.path.join(STYLE_LIBRARY, _img)}')

os.makedirs(STL10_SUB_DIR, exist_ok=True)
print('Downloading sample images ...')
with open('/tmp/stl10_urls.txt', 'w') as _fh:
    _fh.write('\n'.join(
        f'{base_url}/data/04_style_demo/stl10_sub/img_{_i:03d}.jpg'
        for _i in range(100)
    ))
os.system(f'wget -q -i /tmp/stl10_urls.txt -P {STL10_SUB_DIR}')
print('All files ready.')

print('Configuration complete.')

In [ ]:
#@title Demo 4 - roughly applying an image onto a model {display-mode: "form"}
from script.config        import *
from script.vgg_extractor import get_extractor
from script.handlers      import register_handlers
from script.ui_widgets    import ui

get_extractor()
register_handlers(STYLE_LIBRARY, STL10_SUB_DIR)
display(ui)

Disappointing, isn't it? The demo keeps too much detail from the style image, and the result can look like the two pictures layered together. This demo uses a more lightweight method. It reads a style as a set of numbers, then presses those onto your whole photo at once. It can't tell which parts should carry the style and which should stay as they are, so details bleed through. The style-mimicry tools deployed nowdays don't just copy surface textures. They learn the deeper patterns of an artist's work and can reproduce a convincing style from scratch. That power is what makes them controversial. What you see here is a crude demo of those tools.

## Module 6 · Take-Home Messages

- Modern generative models accept intent expressed in ordinary language and translate it into artifact — text, image, audio — without requiring the operator to possess any relevant skill. This is not simply an extension of what earlier models could do. It relocates who can produce creative and knowledge work, and under what conditions.

- Prompting is not a neutral act. The same model, directed differently, produces outputs that differ in register, intent, and implied purpose. Whether that direction constitutes creative authorship is a question courts are beginning to face.

- Model collapse is not caused by bad data. Even clean, accurate training data degrades over generations when a model's own outputs re-enter the training pool — the model grows more confident while becoming less correct (Shumailov et al., *Nature*, 2024).

- Collapse also happens upstream. When automation displaces the entry-level workers through whom expertise is transmitted, the human signal available for training degrades before it reaches any pipeline.

- Style is now extractable at machine scale. The legal premise that style-learning requires human skill and time — and therefore poses no competitive threat — no longer describes what these systems actually do.

- Style can now be extracted from an author's body of work and reproduced on demand at machine scale. Unlike human style-learning, this requires no skill or time, posing a kind of competitive threat to originating artists that may warrant reexamination.

**Remember to disconnect and delete the runtime**

> 1. Click the Runtime menu at the top.
> 2. Click Disconnect and delete runtime.
> 3. Confirm if it asks. That's it — the session is fully closed.